# 04 - Train and register the late-delivery model inside Snowflake

> Generated by `scripts/make_snowflake_notebooks.py`. Run as role `RETAIL_TRANSFORMER` on a CPU compute pool with query warehouse `RETAIL_WH`; **suspend the notebook service when done**.

This trains the v1 logistic regression on `MARTS.MART_LATE_DELIVERY_FEATURES` **here, in Snowflake**, and logs it to the Model Registry as a new version of `RETAIL_OLIST.ML.LATE_DELIVERY_RISK` named `V1WS_<date>_<tag>` (`WS` = trained in a Snowflake workspace). It never replaces a version and never changes the default version. Its quality figures are the out-of-time backtest (`reports/snowflake/late_delivery_metrics.json`): a model trained on every month has no honest held-out score of its own, and its scores for 2018 orders are in-sample.

The last cells score every training order with the new version three ways (in this notebook, `mv.run`, and SQL `MODEL(...)!PREDICT_PROBA`) and compare them, and with the laptop-registered version.

In [ ]:
# Snowflake only. The container runtime ships its own package versions. The results depend on scikit-learn 1.9.1
# (the laptop's version and the registered model's), and the project modules import duckdb at module level.
# Same list as notebooks/snowflake/requirements.txt. If this installs anything, restart the kernel once.
%pip install -q scikit-learn==1.9.1 snowflake-ml-python==2.8.0 duckdb

In [ ]:
import hashlib
import sys
from datetime import UTC, datetime
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path[:0] = [str(ROOT / "src"), str(ROOT / "src" / "ml"), str(ROOT / "snowflake" / "ml")]
import numpy as np  # noqa: E402
import sklearn  # noqa: E402
import register_model as rm  # noqa: E402
from data_access.snowpark import from_active_session  # noqa: E402

db = from_active_session()
print(f"database {db.target.database}, role {db.target.role}, warehouse {db.target.warehouse}")
assert db.target.role == "RETAIL_TRANSFORMER", "run as RETAIL_TRANSFORMER so it owns what it creates"
assert sklearn.__version__ == "1.9.1", f"scikit-learn {sklearn.__version__}: run the install cell, then restart"
session = db.session
session.use_schema(f"{rm.DATABASE}.{rm.SCHEMA}")  # mv.run uploads its input here

In [ ]:
train = rm.training_frame(db)          # labelled orders from 2017 on, read from the dbt mart
model = rm.fit(train)                   # the v1 pipeline from src/ml/late_delivery_baseline.py
sample = rm.parity_sample(train)
metrics = rm.backtest_metrics()
print(f"trained on {len(train):,} orders, late rate {train['is_late'].mean():.3f}")

In [ ]:
# Provenance: the git commit if this workspace has one, otherwise a hash of the model code plus a fingerprint of
# the training data. Either way the version name says exactly what was trained on what.
commit = rm.git_sha()
code_files = sorted((ROOT / "src" / "ml").glob("late_delivery_baseline.py")) + sorted((ROOT / "src" / "data_access").glob("*.py")) \
    + [ROOT / "snowflake" / "ml" / "register_model.py"]
code_hash = hashlib.sha256(b"".join(p.read_bytes() for p in code_files)).hexdigest()
data_fp = db.read_sql("select count(*) as n, max(purchase_ts) as last_ts, hash_agg(*) as h "
                      "from MARTS.MART_LATE_DELIVERY_FEATURES where purchase_ts >= '2017-01-01'").iloc[0]
tag = commit if commit != "unknown" else hashlib.sha256(f"{code_hash}|{data_fp.to_dict()}".encode()).hexdigest()[:7]
version = f"V1WS_{datetime.now(UTC):%Y%m%d}_{tag}".upper()
print("commit", commit, "| code", code_hash[:12], "| data", data_fp.to_dict(), "| version", version)

In [ ]:
from snowflake.ml.registry import Registry

reg = Registry(session=session, database_name=rm.DATABASE, schema_name=rm.SCHEMA)
assert version not in rm.versions(reg), f"{version} already exists; nothing replaced"
mv = reg.log_model(model, **rm.log_kwargs(version, sample, metrics, sklearn.__version__, {
    "training_rows": len(train), "source": "snowflake-workspace-notebook", "git_commit": commit,
    "code_sha256": code_hash, "training_data_rows": int(data_fp["n"]), "training_data_hash": str(data_fp["h"])}))
print("registered", version, "(not set as default)")

In [ ]:
# Three ways to score the same orders with the new version, plus the laptop-registered version.
local = train[["order_id"]].assign(local=model.predict_proba(train[rm.FEATURES])[:, 1])
run = rm.warehouse_parity(mv, model, sample)                       # mv.run on 2,000 orders
where = " where purchase_ts >= '2017-01-01'"
sql_new = db.read_sql(rm.predict_proba_sql(version) + where).rename(columns={"risk_score": "sql_new"})
sql_v1 = db.read_sql(rm.predict_proba_sql("V1_20261113_CC90DC8") + where).rename(columns={"risk_score": "sql_laptop"})
m = local.merge(sql_new, on="order_id").merge(sql_v1, on="order_id")
result = {
    "rows": len(m),
    "mv_run_vs_notebook": run["max_abs_diff"],
    "sql_vs_notebook": float(np.abs(m["sql_new"] - m["local"]).max()),
    "new_vs_laptop_version": float(np.abs(m["sql_new"] - m["sql_laptop"]).max()),
}
print(result)
for k in ("mv_run_vs_notebook", "sql_vs_notebook", "new_vs_laptop_version"):
    mv.set_metric(f"parity_{k}", result[k])
assert len(m) == len(train) and result["mv_run_vs_notebook"] <= 1e-12 and result["sql_vs_notebook"] <= 1e-12